# Notebook 2: Fine-tune

This is the training itself. It loads Qwen3-VL-4B, builds the Kurrent adapter into it, adds a
small new adapter of our own, and trains that new adapter on the line images from Notebook 1. It
scores the 11 dev pages before training and after every round, and keeps the best version.

**This instance needs a GPU** (`ml.g4dn.xlarge` or `ml.g5.xlarge`). README.md explains how to
pick one. Run the cells from top to bottom with **Shift + Enter**.
Kernel: **conda_pytorch_p310** (or whichever `conda_pytorch` kernel your instance lists).

## Step 1. Install the training libraries

Qwen3-VL is newer than the libraries the kernel comes with, so this installs newer versions.
Installed libraries are wiped every time the instance stops, so **run this cell after every
start**. It must run before anything else in the notebook; if you ran other cells first, use
Kernel → Restart Kernel, then start here.

In [ ]:
%pip install -q -r requirements-training.txt

## Step 2. Set up the folders

The same as in Notebook 1. Nothing to change.

In [ ]:
import os
import sys

# Everything this project downloads or creates lives here. It is on the
# notebook instance's storage volume, so it survives Stop and Start.
WORK = "/home/ec2-user/SageMaker/badger-finetune"
DATA = os.path.join(WORK, "data")            # page images, labels, line images
RUNS = os.path.join(WORK, "runs")            # one folder per training run
SCRIPTS = os.path.abspath("scripts")         # the scripts folder next to this notebook
for folder in (WORK, DATA, RUNS):
    os.makedirs(folder, exist_ok=True)
sys.path.insert(0, SCRIPTS)

# Keep the 9 GB model download on the big volume, not the small system disk.
os.environ["HF_HOME"] = os.path.join(WORK, "hf_cache")

SHARED_BUCKET = "ml-marathon-shared"   # the class bucket with the Badger Scribe data
SHARED_PREFIX = "badger-scribe"
print("Working folder:", WORK)

## Step 3. Check the GPU, the disk and the data

This makes sure the instance has a GPU, has room for the model, and that Notebook 1 is done.

In [ ]:
import shutil
import torch

if not torch.cuda.is_available():
    raise SystemExit("This instance has no GPU. Stop it and change its type (see README.md, Part 1).")
gpu = torch.cuda.get_device_properties(0)
bf16 = torch.cuda.get_device_capability()[0] >= 8
print(f"GPU:  {gpu.name}, {gpu.total_memory / 1e9:.0f} GB ({'bf16' if bf16 else 'fp16: no bf16 on this GPU'})")

free = shutil.disk_usage(WORK).free / 1e9
print(f"Disk: {free:.0f} GB free")
if free < 15:
    print("WARNING: under 15 GB free. Stop the instance and make its volume bigger (README.md, Part 1).")

needed = ["train_lines.csv", "dev_pages.csv", "kade_lines"]
missing = [n for n in needed if not os.path.exists(os.path.join(DATA, n))]
if missing:
    raise SystemExit(f"Missing {', '.join(missing)} in {DATA}. Run Notebook 1 first.")
print(f"Data: {sum(1 for _ in open(os.path.join(DATA, 'train_lines.csv'))) - 1} training lines ready")

## Step 4. Choose the run and the settings

There are three kinds of run. Do them in this order:

| `RUN` | What it does | About how long |
|---|---|---|
| `"smoke"` | A tiny run: 64 lines, 1 round, 2 dev pages. It only proves everything works. | 10–15 min the first time (mostly downloading the 9 GB model, which happens once) |
| `"baseline"` | No training. Scores the 11 dev pages with the Kurrent adapter as it is. This is the number to beat. | 5–10 min |
| `"full"` | The real training, with the settings below. | Roughly 30–40 min on g5, 1–1.5 hours on g4dn (estimates until you've timed one) |

The settings only matter for `"full"`. README.md explains each one in plain words ("The settings
you can change"). **Change one at a time**, so you know what made a difference.

Every combination of settings gets its own folder in `runs/`. If you run the same combination
again, it **continues** from its last saved round instead of starting over (handy if something
crashed). To really start it over, delete its folder in `runs/`.

In [ ]:
RUN = "smoke"   # "smoke", then "baseline", then "full"

settings = {
    "lr": 1e-4,           # learning rate: how big each correction is
    "epochs": 3,          # how many times training goes through all the lines
    "lora_r": 16,         # how big our adapter is
    "lora_alpha": 32,     # how strongly it acts; keep it at 2 x lora_r
    "lora_dropout": 0.05, # a little randomness, so it can't just memorise
    "batch_size": 4,      # lines per step; if the GPU runs out of memory, halve this and double grad_accum
    "grad_accum": 4,      # steps per correction; batch_size x grad_accum = 16 lines per correction
    "min_pixels": 65536,  # smallest image size the model sees; higher = more detail, but slower
}
if RUN == "smoke":
    settings.update(epochs=1, max_train_lines=64, dev_pages=2)
elif RUN == "baseline":
    settings = {"epochs": 0}

run_name = RUN if RUN != "full" else "full-" + "-".join(f"{k}{v}" for k, v in settings.items())
run_dir = os.path.join(RUNS, run_name)
for sub in ("model", "checkpoints", "output"):
    os.makedirs(os.path.join(run_dir, sub), exist_ok=True)
print("This run's folder:", run_dir)

## Step 5. Train

The progress appears below as it happens, and is also saved to `train.log` in the run's folder.
What to look for:

- `[dev epoch0] ... mean page CER` is the score **before** training (lower is better).
- `loss` lines show the training error; it should go down.
- After each round there's a new `[dev epochN]` score and, if it's the best so far,
  `best so far ... adapter saved`.

If your browser disconnects or you close the tab, training **keeps going** as long as the instance
runs. To see where it is, open `train.log` in the run's folder (`badger-finetune/runs/<run name>/`)
from the file browser on the left. To stop training, use Kernel → Interrupt Kernel.

In [ ]:
args = {**settings,
        "train_dir": DATA,
        "model_dir": os.path.join(run_dir, "model"),
        "checkpoint_dir": os.path.join(run_dir, "checkpoints"),
        "output_data_dir": os.path.join(run_dir, "output")}
flags = " ".join(f"--{k} {v}" for k, v in args.items())
log = os.path.join(run_dir, "train.log")
python = sys.executable   # this kernel's own Python, where Step 1 installed the libraries
!{python} {SCRIPTS}/finetune_qwen_kurrent_lora.py {flags} 2>&1 | tee -a {log}

## Step 6. How did it go?

"Epoch 0" is the Kurrent adapter **before** our training. The table shows the dev score after
each round. If it goes down and then back up, the later rounds started memorising the training
letters instead of learning to read; the run already kept the best round, and next time you can
use fewer `epochs` or a smaller `lr`.

In [ ]:
import json
import pandas as pd

out = os.path.join(run_dir, "model")
summary = json.load(open(os.path.join(out, "train_summary.json")))
print(f"Kade dev score (CER, lower is better): {summary['dev_cer_before']:.3f} before training, "
      f"{summary['dev_cer_best']:.3f} at best (round {summary['best_epoch']}).")
print("For comparison, the models we used before scored about 0.5.\n")

history = json.load(open(os.path.join(out, "train_log.json")))
curve = pd.DataFrame(history["dev_cer_by_epoch"]).rename(columns={"epoch": "round", "dev_cer": "dev CER"})
print(curve.to_string(index=False))
if history["loss"]:
    import matplotlib.pyplot as plt
    fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.5))
    loss = pd.DataFrame(history["loss"])
    a.plot(loss.step, loss.loss); a.set_xlabel("step"); a.set_ylabel("training loss")
    b.plot(curve["round"], curve["dev CER"], marker="o"); b.set_xlabel("round (epoch)"); b.set_ylabel("dev CER")
    plt.tight_layout(); plt.show()

### Page by page

Each dev page's score before training and at the best round, then the full text of the page that
improved most, so you can see what changed.

In [ ]:
before = pd.read_csv(os.path.join(out, "eval_epoch0.csv"), keep_default_na=False)
best = pd.read_csv(os.path.join(out, f"eval_epoch{summary['best_epoch']}.csv"), keep_default_na=False)
pages = before[["page_id", "cer"]].merge(best[["page_id", "cer"]], on="page_id", suffixes=("_before", "_best"))
pages["improvement"] = pages.cer_before - pages.cer_best
print(pages.round(3).to_string(index=False))

page = pages.sort_values("improvement").iloc[-1].page_id
print(f"\n=== {page} ===\nCORRECT TEXT\n" + before.set_index("page_id").loc[page, "text_ref"][:700])
print("\nBEFORE TRAINING\n" + before.set_index("page_id").loc[page, "text_pred"][:700])
print("\nAFTER TRAINING\n" + best.set_index("page_id").loc[page, "text_pred"][:700])

## Step 7. Save the run to S3 (recommended)

The run's folder is on this instance only. Copy it to your bucket so the team can see it, and so
it's safe if the instance is ever deleted. The trained adapter is about 130 MB; the large
checkpoint files are left out.

In [ ]:
MY_BUCKET = "your-bucket-name"   # <-- your own bucket

!aws s3 sync {run_dir} s3://{MY_BUCKET}/badger-finetune/runs/{run_name} --exclude "checkpoints/*" --quiet
print(f"Saved to s3://{MY_BUCKET}/badger-finetune/runs/{run_name}")

**Next:** write down this run's settings and dev score in the team's results table. Then
either change one setting and run Steps 4 to 7 again, or go to **Notebook 3** to transcribe the
test pages with your best run.

**When you stop for the day, stop the instance** (README.md, "Stopping and starting").